# 06. 실습: Deep Agents로 RAG Agent 조립하기

01~05의 부품을 이번에는 **한 줄씩 직접 이어 붙입니다.** 검색 도구, Agent, Skill, 검증, 기록의 다섯 단계를 만들고, 여섯째로 AutoRAG의 전체 흐름(`ask_two_phase`)을 얹습니다. 마지막에 내 질문 세 개와 바꿔 볼 조건 하나로 끝냅니다. 채점하지 않고 실패한 기록도 결과입니다.

## 학습 목표

- 검색 도구, `create_deep_agent`, Skill 연결, 검증, 기록 읽기의 다섯 단계에 판단 게이트, 질문 분해, 출처 보강이 붙은 전체 흐름을 더해 조립하고, 각 단계가 없으면 무엇이 깨지는지 말한다.
- 대표 사례를 실제 모델로 돌리고, 실패를 검색, 범위, Skill, 답변, 검증, 예산, 판단 중 어느 층의 실패인지 분류한다.
- 조건을 하나만 바꿔 같은 질문을 다시 돌리고, 결과가 그 조건 때문이라고 말할 수 있는 범위를 구분한다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 지도: 01~05 복습과 여섯 조립 단계 | 4 |
| 2 | 1단계: 검색 도구 (융합 검색 + 재순위) | 6 |
| 3 | 왜 Agent에게 맡기는가 | 2 |
| 4 | 2단계: Agent | 4 |
| 5 | Skill을 잇는 네 가지 인자 | 2 |
| 6 | 3단계: Skill 연결 | 4 |
| 7 | 실패 분류법: 어느 층의 실패인가 | 4 |
| 8 | 4단계: 검증 | 3 |
| 9 | 5단계: 기록 읽기 | 4 |
| 10 | 6단계: 전체 흐름 (`ask_two_phase`) | 6 |
| 11 | 대표 사례 실행, 층 분류 연습 | 6 |
| 12 | 통제 실험 규칙과 1회 실행의 한계 | 3 |
| 13 | 직접 해보기: 내 질문 세 개, 조건 하나 | 9 |
| 14 | 정리와 확인 질문 | 3 |


## 호출 규모

- 이 실습은 실제 모델과 Decision API만 씁니다. `.env`에 `ANTHROPIC_API_KEY`와 Decision API 키(`TYPESAFE_API_KEY` 등, `make_decider`가 고릅니다)가 모두 필요합니다. 없으면 첫 코드 셀에서 안내와 함께 멈추고 대신 돌아가는 길은 없습니다.
- 표지: `# [따라하기]` 그대로 실행, `# [바꿔보기]` 값 하나 바꾸기(**먼저 예측**), `# [깨뜨리기]` 일부러 실패, `# [내 것 만들기]` 내 질문.
- `announce`는 실행 직전에 Claude 호출 규모를 알립니다. Decision API 호출은 셀 머리글의 "판단 약 N회"를 보세요. 한 턴에 보통 2~4회이고 `Budget`의 `max_decision_calls`(6)가 한도입니다.
- 저자 실행 1회(비교 셀 `REPEATS = 1`): Claude 호출 48회(입력 약 55만·출력 약 1.8만 토큰, 캐시 읽기·쓰기 포함 약 USD 0.72), Decision API 호출 49회(약 USD 0.0015), 약 4.6분. 반복 2회였던 이전 기록은 67회, USD 1.31입니다. 마지막 비교 셀이 가장 큽니다.


## 1. 지도: 여섯 단계

01~05에서 만든 것을 한 줄로 줄이면 이렇습니다.

| 단계 | 만드는 것 | 어디서 배웠나 |
|---|---|---|
| 1 | 검색 도구 (`@tool`) | 02, 04 |
| 2 | Agent (`create_deep_agent`) | 01, 02 |
| 3 | Skill 연결 | 03 |
| 4 | 검증 (`validate_answer`) | 05 |
| 5 | 기록 읽기 (`budget.report`) | 05 |
| 6 | 전체 흐름 (`ask_two_phase`: 게이트, 분해, 출처 보강) | 04, 05 |

각 단계에서 "이것이 없으면 무엇이 깨지는가"를 생각하세요. 검색 도구가 없으면 찾을 것이 없고, Skill이 없으면 답의 모양이 제멋대로이고, 검증이 없으면 틀린 답이 나가고, 기록이 없으면 이유를 모릅니다.

5단계 끝에서 `ask()` 한 번이 앞 다섯 단계를 대신 한다는 것을 확인하고, 6단계에서 그 위에 AutoRAG의 게이트와 분해를 얹습니다.

In [ ]:
from rag_lab.decisions import make_decider
from rag_lab.settings import announce, key_status, make_embeddings, make_model, make_reranker, require_key, ROOT

for name, value in key_status().items():
    print(f"{name:24s} {value}")
settings = require_key()      # 키가 없으면 한국어 안내와 함께 여기서 멈춘다.
decider = make_decider()      # Decision API 키가 없으면 여기서 멈춘다. Claude로 대신하는 길은 없다.

MAX_CASES = 3                 # 대표 사례 셀에서 실제 모델로 돌릴 사례 수. 올리면 시간과 비용이 사례 수에 비례해 커진다.
model = make_model()


def report_guess(guess, actual):
    """바꿔보기의 예측을 실제 결과와 나란히 찍는다. 판단은 덧붙이지 않는다."""
    print(f"예측: {guess} | 실제: {actual} | 일치 여부: {'일치' if guess == actual else '불일치'}")


print("MAX_CASES =", MAX_CASES, "| 모델:", model.model, "| 판단(Decision API):", decider.name)


## 2. 1단계: 검색 도구

일반 함수에 `@tool`을 붙이면 이름, 설명, 인자를 가진 Agent의 도구가 됩니다. 먼저 사내 안내문 세 줄로 가장 작은 검색 도구를 만듭니다. 04의 `BM25Index`와 `Chunk`를 씁니다. 모델 호출은 없습니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
from langchain_core.tools import tool
from rag_lab.corpus import Chunk
from rag_lab.retrieval import BM25Index

NOTES = ["사내 식당은 오전 11시 30분에 문을 엽니다.", "도서관 책은 2주 동안 빌릴 수 있습니다.", "통근 버스는 오후 6시에 출발합니다."]
index = BM25Index([Chunk(f"n{i}", "notes", "", s, 0, len(s), 1, 1) for i, s in enumerate(NOTES)])


@tool
def search_notes(query: str, limit: int = 1) -> list:
    """사내 안내문에서 질의와 맞는 문장 번호와 점수를 찾는다."""
    return index.search(query, limit=limit)


print(search_notes.invoke({"query": "식당 언제 열어요?"}))

출력의 `n0`은 첫 번째 문장이고 숫자는 점수입니다. 함수 위의 설명 글(docstring)이 Agent가 읽는 도구 설명이 됩니다. 이제 값 하나를 바꿉니다. **먼저 예측을 적고** 실행하세요.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회. 바꿀 값: limit를 1에서 3으로.
# 예측(먼저 적기): limit=3이면 결과가 몇 개 나올까? 문장이 세 개뿐이니 3개일까? 숫자로 적습니다.
MY_GUESS = 3
found = search_notes.invoke({"query": "식당 언제 열어요?", "limit": 3})
print(found)
report_guess(MY_GUESS, len(found))
print("점수가 0보다 큰 조각만 나옵니다. 질의와 겹치는 단어가 없는 문장은 후보가 아닙니다.")


같은 모양을 업무 문서로 옮깁니다. 달라지는 점은 세 가지입니다. 검색기가 `Retriever(load_corpus(), embeddings=..., mode="fusion")`로 바뀌고(BM25와 dense 점수를 합칩니다), 도구는 후보를 넓게 모은 뒤 `make_reranker()`의 교차 인코더로 **질문과 조각을 한 쌍씩 다시 채점**해 순위를 정하고, 도구에 **고객과 기준일을 인자로 두지 않습니다.** 고객과 기준일은 `RetrievalContext`가 들고 있는 호출자의 값입니다.

넓은 후보 + 재순위란, 받은 검색어로 최대 50개 조각을 먼저 모은 뒤(`pipeline.retrieve`의 `pool`) 그 후보만 재순위 모델이 채점해 상위 몇 개만 근거로 돌려주는 방식입니다. 재순위가 실패하면 융합 순서를 그대로 쓰고 기록에 `rerank-failed` 경고를 남깁니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회
from datetime import date
from rag_lab.corpus import Scope, load_corpus
from rag_lab.retrieval import Retriever
from rag_lab.evidence import RetrievalContext, make_search_tools

embeddings = make_embeddings()
reranker = make_reranker()
search_index = Retriever(load_corpus(), embeddings=embeddings, mode="fusion")
scope = Scope(entity="알파", as_of=date(2026, 9, 8))
context = RetrievalContext(search_index, scope, reranker=reranker)  # 도구가 부르는 검색은 pool 50 후보를 재순위한다
search_tool, read_tool = make_search_tools(context)

result = search_tool.invoke({"query": "서비스 크레딧 신청 기한"})
print("status:", result["status"], "| related_docs:", result["related_docs"], "| sufficiency:", result["sufficiency"]["coverage"])
for item in result["evidence"]:
    print(f"  {item['evidence_id']}  {item['doc_id']}  L{item['start_line']}-{item['end_line']}  {item['quote'][:40]!r}")
print("도구 인자:", list(search_tool.args))
warnings = [e["warning"] for e in context.budget.report()["events"] if e["event"] == "retrieval_warning"]
print("재순위 경고:", warnings or "없음")

`evidence`는 원문 인용과 좌표(문서, 행), `related_docs`는 같은 계열인데 못 찾은 문서, `sufficiency`는 "근거가 모자랄 수 있다"는 신호입니다. 도구 인자에 고객이 없다는 점과 재순위 경고가 비었는지 확인하세요. 재순위는 자르지 않으므로 `limit=6`을 채우느라 상관없는 문서(위 출력의 `travel`)가 들어올 수 있습니다. 이번에는 도구를 일부러 잘못 불러 봅니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회. 세 가지 실패는 서로 다른 모양이다.
from rag_lab.evidence import Budget, BudgetExceeded

print("1) 빈 검색어   :", search_tool.invoke({"query": ""}))
print("2) 범위 밖 문서:", read_tool.invoke({"doc_id": "sla-b"}))
try:
    RetrievalContext(search_index, scope, Budget(max_searches=0)).search("서비스 크레딧")
except BudgetExceeded as error:
    print("3) 예산 0회    :", type(error).__name__, error)
print("앞의 둘은 오류 없이 상태값으로 돌아오고, 셋째는 예외로 실행을 멈춥니다. 왜 다르게 만들었을까요?")

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회. 바꿀 값: 고객을 "알파"에서 "베타"로(질의는 그대로).
# 예측(먼저 적기): 알파 검색에서 나온 근거 문서와 겹치는 문서가 있을까? "겹친다" 또는 "겹치지 않는다"로 적습니다.
MY_GUESS = "겹치지 않는다"
beta_context = RetrievalContext(search_index, Scope(entity="베타", as_of=date(2026, 9, 8)), reranker=reranker)
beta_search, _ = make_search_tools(beta_context)
beta_result = beta_search.invoke({"query": "서비스 크레딧 신청 기한"})
alpha_docs = {e["doc_id"] for e in result["evidence"]}
beta_docs = {e["doc_id"] for e in beta_result["evidence"]}
print("알파 근거 문서:", sorted(alpha_docs), "| 베타 근거 문서:", sorted(beta_docs))
report_guess(MY_GUESS, "겹친다" if alpha_docs & beta_docs else "겹치지 않는다")


예측과 달리 겹쳤다면 문서 이름을 보세요. 고객 범위는 해당 고객 문서와 모든 고객이 쓰는 `공통` 문서를 허용해 `travel`이 양쪽에 들어왔습니다. 고객을 가르는 것은 범위 필터입니다.

## 3. 왜 Agent에게 맡기는가

검색 도구는 "한 번 찾는 일"만 합니다. 몇 번 어떤 단어로 찾고 어떤 문서를 읽을지는 **모델이 정하는 반복**이고, 그 틀이 Deep Agents입니다. 반복이 길수록 비용이 늘어 예산이 필요합니다. Agent에는 모델 호출 수, 검색 수, 시간 한도가 붙고, 닿으면 조용히 줄이지 않고 예외로 멈춥니다.

## 4. 2단계: Agent

`create_deep_agent(model=..., tools=..., system_prompt=...)` 한 줄이면 Agent가 됩니다. Skill도 검증도 없는 가장 단순한 Agent이고, 메시지를 줄별로 읽습니다. 모델 호출 약 3회입니다. `recursion_limit`은 반복 상한이고 넘으면 `GraphRecursionError`로 멈춥니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회
from deepagents import create_deep_agent
from langgraph.errors import GraphRecursionError

QUESTION = "알파 서비스 크레딧 신청 기한은 언제인가요?"


def run_step2(step_scope):
    step_context = RetrievalContext(search_index, step_scope, reranker=reranker)
    tools = make_search_tools(step_context)
    step2_prompt = (f"당신은 문서 근거를 확인하는 한국어 Agent입니다. 고객={step_scope.entity}, 기준일={step_scope.as_of.isoformat()}.\n"
                    "search_knowledge로 근거를 찾아 답하세요.")
    agent = create_deep_agent(model=model, tools=tools, system_prompt=step2_prompt)
    announce("2단계 Agent", model_calls=3)
    try:
        step_result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]}, config={"recursion_limit": 20})
    except (BudgetExceeded, GraphRecursionError) as error:
        print("실행이 멈췄습니다:", type(error).__name__, error)
        return step_context
    for message in step_result["messages"]:
        calls = [(c["name"], c["args"].get("query")) for c in getattr(message, "tool_calls", [])]
        print(f"[{message.type:5s}] {str(message.text)[:90]!r} {calls or ''}")
    return step_context


step2_context = run_step2(scope)
print("검색 횟수:", step2_context.search_count, "| 모은 근거 문서:", sorted({e.doc_id for e in step2_context.evidence.values()}))

`[ai]` 줄의 괄호가 모델이 부른 도구와 검색어입니다. 질문을 그대로 검색했는지, 다시 썼는지 보세요. 이제 기준일 하나만 바꿉니다. 문서에 개정 이력이 있어 기준일에 따라 "그날 유효한 문서"가 달라집니다. 검색 결과와 답이 그 차이를 반영하는지 보세요.

In [ ]:
# [바꿔보기] 유료, 모델 호출 약 3회. 바꿀 값: 기준일을 2026-09-08에서 2026-07-15로.
# 예측(먼저 적기): 근거 문서 목록이 바뀔까? "바뀐다" 또는 "안 바뀐다"로 적습니다.
MY_GUESS = "바뀐다"
july_context = run_step2(Scope(entity="알파", as_of=date(2026, 7, 15)))
sept_docs = sorted({e.doc_id for e in step2_context.evidence.values()})
july_docs = sorted({e.doc_id for e in july_context.evidence.values()})
print("9월 근거 문서:", sept_docs, "| 7월 근거 문서:", july_docs)
report_guess(MY_GUESS, "바뀐다" if sept_docs != july_docs else "안 바뀐다")


## 5. Skill을 잇는 네 가지 인자

03에서 Skill은 폴더 안의 SKILL.md였습니다. Agent에게 알려 주려면 인자가 필요합니다.

- `backend`와 `skills=["/"]`: Skill 폴더 위치. `TrackedSkillsBackend`가 `skills/`를 읽고 **SKILL.md를 끝까지 읽었을 때만** 읽었다고 기록합니다.
- `permissions`: 쓰기를 모두 막습니다. 문서를 읽는 Agent가 파일을 고치면 안 됩니다.
- `response_format=ProviderStrategy(AnswerPayload)`: 최종 답을 정해진 모양(Skill, 상태, 절, 인용)으로 받습니다.

여기에 `allowlist_middleware`가 허용 도구만 보이게 하고 호출 수를 셉니다. 시스템 프롬프트는 Skill을 읽으라고 지시하는 `SYSTEM_PROMPT`로 바꿉니다.

## 6. 3단계: Skill 연결

**새 턴이므로 `RetrievalContext`를 새로 만듭니다.** 앞 단계의 `evidence`와 검색 횟수를 물려받으면 검증이 거짓으로 통과합니다. 모델 호출 약 5회이고, 이 셀의 `result`를 뒤 단계에서 계속 씁니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 5회
from deepagents.middleware.filesystem import FilesystemPermission
from langchain.agents.structured_output import ProviderStrategy
from rag_lab.agent import SYSTEM_PROMPT, TrackedSkillsBackend, allowlist_middleware
from rag_lab.answer import AnswerPayload

context = RetrievalContext(search_index, scope, reranker=reranker)       # 새 턴: 앞 단계의 근거를 물려받지 않는다
agent = create_deep_agent(
    model=model, tools=make_search_tools(context),
    system_prompt=SYSTEM_PROMPT.format(entity=scope.entity, as_of=scope.as_of.isoformat()),
    backend=TrackedSkillsBackend(ROOT / "skills", context.loaded_skills.add), skills=["/"],
    permissions=[FilesystemPermission(operations=["write"], paths=["/**"], mode="deny")],
    response_format=ProviderStrategy(AnswerPayload),
    middleware=allowlist_middleware(context.budget),
)
announce("3단계 Skill 연결", model_calls=5)
result = None
try:
    result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]}, config={"recursion_limit": 40})
except (BudgetExceeded, GraphRecursionError) as error:
    print("실행이 멈췄습니다:", type(error).__name__, error)
answer = result["structured_response"] if result else None
print("읽은 Skill:", sorted(context.loaded_skills), "| 검색 횟수:", context.search_count)
print("답변 상태:", answer.status if answer else "답변 없음", "| 답변 Skill:", answer.skill if answer else "-")

`loaded_skills`에 이름이 있으면 그 Skill의 SKILL.md를 끝까지 읽은 것이고, 답변 `skill`은 읽은 Skill 중 하나여야 합니다. 헷갈리기 쉬운 Skill이 하나 있어 일부러 부딪혀 봅니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회. 읽을 것: 어떤 오류이고, 왜 이 폴더만 그런가.
from rag_lab.answer import skill_spec

try:
    skill_spec("grounded-answer")
except ValueError as error:
    print("ValueError:", error)
print("답변 Skill 목록:", sorted(p.parent.name for p in (ROOT / "skills").glob("*/output.json")))
print("grounded-answer 폴더에는 output.json이 없습니다. 검색 안내 Skill이고, 답변 Skill이 아닙니다.")

## 7. 실패 분류법: 어느 층의 실패인가

Agent가 틀렸을 때 "모델이 못한다"로 끝내면 고칠 곳을 알 수 없습니다. 일곱 층 중 어디인지 먼저 가립니다.

- **검색**: 필요한 문서가 근거에 없다. `rerank-failed` 경고가 남았다면 이 층의 단서다.
- **범위**: 다른 고객, 다른 기준일의 문서를 썼거나 막았다.
- **Skill**: 읽지 않았거나 잘못 골랐다.
- **답변**: 근거가 있는데 포기했거나, 없는데 답했다.
- **검증**: 인용 ID, 숫자, 절 구조가 검사를 통과하지 못했다.
- **예산**: 한도에 걸려 실행이 멈췄다.
- **판단(Decision API)**: 게이트가 응답을 못 받아 기본값을 썼다. `gates`에 `fallback`이 남고, 오류가 아니라 `status`는 `answered`일 수 있다.

층 후보는 판정이 아니라 "여기부터 보라"는 표시입니다. "근거가 없는데 답했다"는 검색의 잘못일까요, 답변의 잘못일까요? 검색이 아무것도 못 찾았는데 지어냈다면 답변 층이고, 문서가 범위 필터에 막혔다면 범위 층입니다.

## 8. 4단계: 검증

모델의 답은 입력이지 사실이 아닙니다. `validate_answer`는 인용 ID가 이번 턴에 모은 근거인지, 숫자가 인용문에 있는지, 절 구조가 Skill과 같은지를 코드로 검사합니다. 새 호출은 없습니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 (앞 단계의 같은 실행을 쓴다)
from rag_lab.answer import render_answer, validate_answer

if answer is None:
    print("앞 단계에서 답변을 얻지 못했습니다. 검증할 답이 없습니다.")
else:
    errors = validate_answer(answer, context, QUESTION)
    print("검증 오류:", errors or "없음")
    print()
    print(render_answer(answer, context))
    print()
    for item in context.evidence.values():
        print(f"  {item.evidence_id}  {item.doc_id}  L{item.start_line}-{item.end_line}  v{item.version}")

오류 목록이 비어 있어도 "맞다"가 아니라 "`validate_answer`의 결정적 검사에서 걸리지 않았다"입니다. 이제 손으로 답을 망가뜨려 검사가 잡는지 봅니다.

In [ ]:
# [깨뜨리기] 무료, 모델 호출 0회. 첫 인용을 없는 ID로 바꾼다. 읽을 오류: '현재 턴에 없는 인용 ID'
if answer is None or not answer.sections or not answer.sections[0].items:
    print("망가뜨릴 답변이 없습니다. 앞 단계 결과를 먼저 확인하세요.")
else:
    first_section = answer.sections[0]
    fake_item = first_section.items[0].model_copy(update={"evidence_ids": ["E-없는ID"]})
    fake_section = first_section.model_copy(update={"items": [fake_item, *first_section.items[1:]]})
    broken = answer.model_copy(update={"sections": [fake_section, *answer.sections[1:]]})
    for error in validate_answer(broken, context, QUESTION):
        print("-", error)

검사는 인용 ID와 숫자를 봅니다. 숫자 없는 거짓은 못 잡아서 "이 문장이 근거로 지지되는가"를 모델에게 묻는 `review_support`가 따로 있고, 검토 모델도 틀릴 수 있습니다. 앞 [깨뜨리기]에서 숫자 오류가 따라 나온 것은 숫자가 든 인용문과 문장의 연결이 끊겼기 때문입니다.

In [ ]:
# [바꿔보기] 유료, 모델 호출 2회. 바꿀 값: review_support에 넘기는 question을 다른 사실을 묻는 질문으로.
# 예측(먼저 적기): 신청 기한 답변을 '월간 가용률 비율' 질문에 대해 검토시키면 supported가 무엇일까? True 또는 False.
from rag_lab.answer import review_support

MY_GUESS = False
OTHER_QUESTION = "알파 SLA에서 가용률이 99.0% 미만일 때 서비스 크레딧 비율은 얼마인가요?"
if answer is None:
    print("검토할 답변이 없습니다.")
else:
    announce("지지 검토", model_calls=2)
    same = review_support(model, QUESTION, answer, context)
    other = review_support(model, OTHER_QUESTION, answer, context)
    print("원래 질문  supported:", same.supported, same.reasons)
    print("다른 질문  supported:", other.supported, other.reasons)
    report_guess(MY_GUESS, other.supported)


위 셀의 예측과 결과를 비교하세요. 이번 저자 실행은 원래 질문 `True`, 다른 질문 `False`였고, 사유에 99.0% 크레딧 비율이 근거에 없다고 적혀 있었습니다. 모델이 검토하므로 판정은 실행마다 달라질 수 있습니다. 값만 믿지 말고 출력된 사유를 읽으세요. 통과해도 동문서답일 수 있습니다.

## 9. 5단계: 기록 읽기

Agent가 한 일은 기록에만 남습니다. `budget.report()`에 호출 횟수와 순서가 있습니다. 여기에 실행마다 다른 `run_id`(`uuid4`)를 붙여 저장하고 다시 읽습니다. 읽을 것은 호출 횟수(`counts`), 사건 순서, 검색어, 걸린 시간, 그리고 키 값이 없다는 것입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회 (앞 단계의 같은 실행을 쓴다)
import json
from uuid import uuid4
from rag_lab.agent import write_run_record

trace = context.budget.report()
my_record = {"run_id": uuid4().hex, "question": QUESTION, "trace": trace, "loaded_skills": sorted(context.loaded_skills)}
saved_path = write_run_record(my_record)
reread = json.loads(saved_path.read_text(encoding="utf-8"))

print("기록 파일:", saved_path.relative_to(ROOT))
print("counts:", reread["trace"]["counts"], "| 걸린 시간: %.1f초" % reread["trace"]["elapsed_seconds"])
for event in reread["trace"]["events"][:12]:
    detail = {k: v for k, v in event.items() if k not in ("event", "seconds")}
    print(f"  {event['seconds']:6.2f}s {event['event']:12s} {str(detail)[:60]}")
print("키 형식 문자열이 있는가:", "sk-" in saved_path.read_text(encoding="utf-8"))

지금까지 손으로 조립한 다섯 단계를 `ask()`가 한 번에 합니다. 돌려주는 기록의 키가 손으로 만든 값과 어떻게 대응하는지 봅니다. 모델 호출 약 5회입니다. `ask`에는 게이트가 없습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 약 5회
from rag_lab.agent import ask

announce("ask 한 번", model_calls=5)
record = ask(QUESTION, retriever=search_index, scope=scope, model=model, budget=Budget(), reranker=reranker)
print("반환 키:", sorted(record))
print("status:", record["status"], "| validation:", record["validation"], "| errors:", record["errors"] or "없음")
print("loaded_skills :", record["loaded_skills"], "   <- 3단계의 context.loaded_skills")
print("trace.counts  :", record["trace"]["counts"], "   <- 5단계의 budget.report()")
print("evidence      :", [e["doc_id"] for e in record["evidence"]], "   <- context.evidence")
print("markdown      :", record["markdown"][:60].replace("\n", " "), "...   <- 4단계의 render_answer")

`ask`는 실패해도 예외 없이 `status="error"` 기록을 돌려주고 파일에 남깁니다. 예산을 일부러 모자라게 해서 확인합니다. 예산을 올려 실패를 감추지 않습니다.

In [ ]:
# [깨뜨리기] 유료, 모델 호출 약 1회. 읽을 것: 예외 없이 status='error', 실패 행도 파일에 남는다.
announce("예산 모자란 ask", model_calls=1)
tiny = ask(QUESTION, retriever=search_index, scope=scope, model=model, budget=Budget(max_model_calls=1), max_attempts=1, reranker=reranker)
print("status:", tiny["status"], "| errors:", tiny["errors"])
print("실패 기록 파일이 있는가:", (ROOT / "outputs" / "runs" / f"{tiny['run_id']}.json").exists())

## 10. 6단계: 전체 흐름

이 과정의 전체 흐름은 `ask_two_phase`(`rag_lab/phases.py:125`)입니다.

1. **분해 게이트**: 검색을 여러 번 해야 하는 질문인지 Decision API가 확률로 답합니다. .5 이상이면 `decompose_question`이 질의를 나눕니다. 분해는 기본으로 켜져 있고 질문마다 게이트가 정합니다.
2. **미리 검색**: 모든 질의의 후보를 합쳐 원 질문으로 재순위합니다. 출처 보강도(corroboration)가 켜져 있어 같은 출처의 조각이 여럿이면 점수를 더합니다.
3. **빠른 답변**: 도구 없이 미리 찾은 근거와 Skill 본문만으로 씁니다. `loaded_skills`는 읽은 Skill이 아니라 실은 Skill입니다.
4. **인용 관련성 게이트**: 인용마다 무관한지 묻고 .3 미만을 지웁니다.
5. **추가 확인 게이트**: 답이 빠졌거나 불확실한지 묻습니다. 참이거나 빠른 답이 검증에 걸리면
6. **검증 턴**: `build_rag_agent`(`rag_lab/phases.py:107`)가 3단계와 같은 조립으로 Agent를 만들어 원문을 다시 읽고 고칩니다.

Decision API가 응답하지 못하면 게이트는 AutoRAG의 기본값(분해 안 함, 검증함, 인용 유지)을 쓰고 `fallback`을 남깁니다. 확률은 보정되지 않았고 문턱값은 코드 규칙입니다. 문서(`docs/retrieval_benchmarks.md`)는 수업 문서 48문항 중 5문항에서 분해 게이트가 열렸다고 적었습니다(한 번 실행). 먼저 예측하고 실행하세요.

In [ ]:
# [따라하기] 유료, 모델 호출 약 3회 + 판단(Decision API) 약 3회. 전체 흐름을 한 질문에 한 번 돌린다.
from rag_lab.phases import ask_two_phase


def show_flow(rec):
    """ask_two_phase 기록에서 질의, 게이트별 확률과 결정, 검증 턴 여부, 호출 수를 읽어 찍는다."""
    print("  질의:", rec["queries"])
    for gate in rec["gates"]:
        if gate["gate"] == "citation_relevance":
            lowest = min(gate["probabilities"].values(), default=None)
            result = f"인용 {len(gate['probabilities'])}개 판단, 최저 P={lowest} -> 뺀 인용 {gate['decision']}"
        else:
            result = f"P={gate['probability']} (문턱값 {gate['threshold']}) -> {'참' if gate['decision'] else '거짓'}"
        print(f"  게이트 {gate['gate']:20s} {result}" + (f"  [대체값: {gate['fallback']}]" if gate["fallback"] else ""))
    counts = rec["trace"]["counts"]
    print(f"  검증 턴: {'돌림' if rec['verification'] == 'ran' else '건너뜀'} | status: {rec['status']} | 검증: {rec['validation']} | "
          f"모델 호출 {counts.get('model', 0)} | 판단 호출 {counts.get('decision', 0)} | {rec['trace']['elapsed_seconds']:.1f}초")


announce("6단계 전체 흐름", model_calls=4)
flow = ask_two_phase(QUESTION, retriever=search_index, scope=scope, model=model, decider=decider, reranker=reranker)
show_flow(flow)
print("진단:", flow["diagnostics"] or "없음", "| 근거 문서:", sorted({e["doc_id"] for e in flow["evidence"]}))


게이트마다 확률과 문턱값이 찍힙니다. 검증 턴이 "건너뜀"이면 빠른 답변이 그대로 나간 것이고, "돌림"이면 `build_rag_agent`의 Agent가 한 번 더 일했습니다. 같은 질문도 게이트 결정이 갈릴 수 있습니다(문서는 추가 확인 확률이 .82와 .45로 갈린 예를 적었습니다).

## 11. 대표 사례 실행

질문 유형별 첫 사례를 `MAX_CASES`개 골라 `ask_two_phase`로 돌립니다. **정답표는 사례를 고르는 데 쓰지 않고**, 돌린 뒤 층 후보를 붙일 때만 `load_gold()`를 읽습니다.

In [ ]:
# [따라하기] 유료, 모델 호출 사례당 약 3회 + 판단(Decision API) 사례당 약 3회
from rag_lab.corpus import load_cases
from rag_lab.evaluation import load_gold

cases = load_cases()
representatives = {}
for case in cases:
    representatives.setdefault(case["type"], case)      # 정답표를 보지 않고 유형별 첫 사례를 고른다
picked = list(representatives.values())[:MAX_CASES]
print(f"유형 {len(representatives)}개 중 {len(picked)}개 실행 (MAX_CASES={MAX_CASES}):", [c["id"] for c in picked])


def scope_of(case):
    return Scope(entity=case["entity"], as_of=date.fromisoformat(case["as_of"]))


def cited_docs(rec):
    by_id = {e["evidence_id"]: e["doc_id"] for e in rec["evidence"]}
    ids = [i for s in (rec["answer"] or {}).get("sections", []) for item in s["items"] for i in item["evidence_ids"]]
    return sorted({by_id[i] for i in ids if i in by_id})


announce("대표 사례 실행", model_calls=4 * len(picked))
results = [(case, ask_two_phase(case["question"], retriever=search_index, scope=scope_of(case), model=model, decider=decider, reranker=reranker))
           for case in picked]

gold = load_gold()          # 돌린 뒤에만 읽는다
for case, rec in results:
    expected = gold[case["id"]]
    print(f"{case['id']:28s} {rec['status']:13s} 검증 {rec['validation']:7s} 인용 문서 {cited_docs(rec)} / 기대 문서 {expected['expected_docs']}")
    show_flow(rec)
    if rec["errors"]:
        print("    오류:", rec["errors"])


한 줄이 한 번의 관찰이고 실패한 행도 남깁니다. 첫 사례는 추가 확인 게이트가 .14로 닫혔는데 검증 턴이 돌았습니다. 빠른 답이 `인용문에 없는 숫자: 20`으로 검증에 걸렸기 때문입니다(한 번 실행). 행마다 층 후보를 규칙으로 뽑습니다. 판정이 아니라 "여기부터 보라"는 표시입니다.

In [ ]:
# [따라하기] 무료, 모델 호출 0회. 위 표의 층 후보를 붙인다.
def failure_candidates(rec, expected_status, expected_docs=()):
    layers = []
    scope = Scope(entity=rec["scope"]["entity"], as_of=date.fromisoformat(rec["scope"]["as_of"]))
    if any(not search_index.documents[doc].applies(scope) for doc in expected_docs):
        layers.append("범위")
    elif expected_docs and not set(expected_docs) <= {e["doc_id"] for e in rec["evidence"]}:
        layers.append("검색")
    if not rec["loaded_skills"] or any("SKILL.md" in e for e in rec["errors"]):
        layers.append("Skill")
    if any(e.get("warning", "").startswith("rerank-failed") for e in rec.get("trace", {}).get("events", [])):
        layers.append("검색(재순위 실패)")
    if any(gate["fallback"] for gate in rec.get("gates", [])):
        layers.append("판단(Decision API 대체값)")
    if rec["status"] == "error":
        rest = [e for e in rec["errors"] if "SKILL.md" not in e]
        if any("한도" in e for e in rest):
            layers.append("예산")
        elif any(e.split(":")[0].endswith("Error") for e in rest):
            layers.append("실행")
        elif rest:
            layers.append("검증")
    if rec["status"] == "answered" and expected_status == "insufficient":
        layers.append("답변(근거 부족인데 답함)")
    if rec["status"] == "insufficient" and expected_status == "answered":
        layers.append("답변(답할 수 있는데 포기)")
    return layers or ["후보 없음"]


for case, rec in results:
    expected = gold[case["id"]]
    status = "insufficient" if expected["abstain"] else "answered"
    print(f"{case['id']:28s} 층 후보 {failure_candidates(rec, status, expected['expected_docs'])}")

표가 모두 `후보 없음`이면 연습할 재료가 없습니다. 아래에 일곱 층과 재순위 실패 경고가 하나씩 나오는 **손으로 만든 기록**(모델 출력이 아닙니다)을 둡니다. 결과가 늘 같습니다. 사례마다 층을 먼저 적고 실행해 비교하세요. 범위 사례는 후보가 둘 나옵니다. 범위를 고치지 않으면 답변을 아무리 고쳐도 근거를 쓸 수 없으니 범위부터 봅니다.

In [ ]:
# [바꿔보기] 무료, 모델 호출 0회. 바꿀 값: MY_GUESSES에 사례마다 층을 먼저 적는다. 아래 기록은 손으로 만든 것이다.
MY_GUESSES = {"문서 빠짐": "", "다른 고객": "", "다른 Skill 읽음": "", "근거 없이 답함": "", "연결 틀림": "", "호출 초과": "", "재순위 실패": "", "판단 대체값": ""}


def handmade_record(**fields):
    return {"status": "answered", "errors": [], "loaded_skills": ["grounded-qa"], "evidence": [{"doc_id": "sla-a-v2"}],
            "scope": {"entity": "알파", "as_of": "2026-09-08"}, **fields}


PRACTICE = {
    "문서 빠짐": (handmade_record(), "answered", ["sla-a-v2", "sla-a-amendment"]),
    "다른 고객": (handmade_record(status="insufficient", evidence=[], scope={"entity": "베타", "as_of": "2026-09-08"}), "answered", ["sla-a-v2"]),
    "다른 Skill 읽음": (handmade_record(status="error", errors=["grounded-qa/SKILL.md 전체 본문을 read_file로 읽어야 합니다."], loaded_skills=["procedure"]), "answered", ["sla-a-v2"]),
    "근거 없이 답함": (handmade_record(evidence=[]), "insufficient", []),
    "연결 틀림": (handmade_record(status="error", errors=["인용문에 없는 숫자: 20%"]), "answered", ["sla-a-v2"]),
    "재순위 실패": (handmade_record(trace={"events": [{"event": "retrieval_warning", "warning": "rerank-failed: OSError"}]}), "answered", ["sla-a-v2"]),
    "판단 대체값": (handmade_record(gates=[{"gate": "follow_up", "fallback": "follow-up-check-fallback", "reason": "decision: HTTP 503"}]), "answered", ["sla-a-v2"]),
    "호출 초과": (handmade_record(status="error", errors=["BudgetExceeded: model 호출 한도를 초과했습니다."], evidence=[]), "answered", []),
}
for name, (rec, expected_status, expected_docs) in PRACTICE.items():
    print(f"{name:14s} 내 예측: {MY_GUESSES[name] or '(비어 있음)':12s} 층 후보: {failure_candidates(rec, expected_status, expected_docs)}")


## 12. 통제 실험 규칙과 1회 실행의 한계

조건을 바꿔 비교할 때 지킬 규칙은 셋입니다.

1. **하나만 바꿉니다.** 둘을 같이 바꾸면 어느 쪽 때문인지 모릅니다.
2. **같은 사례로 비교합니다.**
3. **전과 후를 모두 보입니다.** 나아진 행만 고르지 않습니다.

같은 질문도 모델이 다른 검색어를 쓰고 게이트 결정이 갈릴 수 있습니다. 마지막 비교 셀은 전과 후를 **같은 셀에서 번갈아** `REPEATS`번씩 돌립니다. 비용 때문에 기본값은 `REPEATS = 1`이라 노이즈와 효과를 가를 수 없습니다. 낼 수 있으면 2로 바꿔 같은 질문이 달라지는지 보세요.

- 2회 이상이면 흔들림이 조건 차이와 비슷한지 봅니다.
- 1회로는 흔들림을, 2회로도 평균, 신뢰 구간, 우연일 확률을 못 봅니다.

## 직접 해보기

두 셀을 차례로 돌립니다. 예시 값으로 돌린 뒤 **자기 질문으로 바꿔** 다시 돌리세요.

1. 내 질문 세 개: 답할 수 있는 질문, 문서에 없는 질문, 범위 함정 질문(다른 고객을 묻는 질문)과 각각의 기대 상태(`answered` 또는 `insufficient`). 빈 문자열이면 뺍니다.
2. 조건 하나: `chunker`, `mode`, `k`, `reranker`, `decompose`, `corroboration` 중 정확히 하나. 기준선은 지금까지의 설정(융합 검색, 재순위, `k=3`, 분해 게이트 켬, 보강도 켬)입니다. 검색 표는 04의 어려운 질문 36개(정답표는 초안)로 재고, 놓친 행은 **전과 후 모두 전부** 찍고 범위 밖 정답 행은 따로 셉니다. 이어 세 질문을 전후 `REPEATS`번씩(기본 1) 번갈아 돌립니다. 예측을 먼저 적으세요.

**구조적 한계를 예측에 넣으세요.** 수업 문서는 조각 32개, 후보 50개라 범위 안 조각이 모두 후보에 들어가고 재순위가 그 전체를 줄 세웁니다. 재순위가 켜져 있으면 분해나 보강도를 바꿔도 검색 순위는 바뀔 수 없습니다. 보강도의 효과를 보려면 `BASE`의 `reranker`를 `"off"`로 바꾼 뒤 `corroboration`을 끄세요(기준선 변경이니 적어 둡니다). 분해는 검색 표에 잡히지 않고 `ask_two_phase` 비교에서만 보입니다.

In [ ]:
# [내 것 만들기] 유료, 모델 호출 약 12회 + 판단(Decision API) 약 9회. 아래 예시를 자기 질문으로 바꿔 보세요. 질문을 ""로 두면 빠집니다.
MY_ENTITY = "알파"
MY_AS_OF = "2026-09-08"
MY_QUESTIONS = [
    ("답할 수 있는 질문", "서버가 이상해서 누군가를 깨워야 하는 기준 수치가 뭔가요?", "answered"),
    ("문서에 없는 질문", "알파 고객 전담 매니저의 휴대전화 번호는 무엇인가요?", "insufficient"),
    ("범위 함정 질문", "알파 계약 기준으로 베타 서비스의 장애 응답 시간을 알려주세요.", "insufficient"),
]

my_items = [item for item in MY_QUESTIONS if item[1].strip()]
my_scope = Scope(entity=MY_ENTITY, as_of=date.fromisoformat(MY_AS_OF))
my_results = []                                  # [(라벨, 질문, 기대 상태, 기록)]
if not my_items:
    print("질문이 모두 비어 있습니다. MY_QUESTIONS에 질문을 하나 이상 적고 다시 실행하세요.")
    print("예: ('답할 수 있는 질문', '알파 SLA 크레딧은 어떻게 신청하나요?', 'answered')")
else:
    announce("내 질문", model_calls=4 * len(my_items))
    for label, question, expected_status in my_items:
        rec = ask_two_phase(question, retriever=search_index, scope=my_scope, model=model, decider=decider, reranker=reranker)
        my_results.append((label, question, expected_status, rec))
        print(f"[{label}] 기대 {expected_status} -> status={rec['status']} 검증={rec['validation']} "
              f"인용={cited_docs(rec)} 층 후보={failure_candidates(rec, expected_status)}")
        show_flow(rec)
        if rec["errors"]:
            print("    오류:", rec["errors"])


In [ ]:
# [내 것 만들기] 유료, 모델 호출 최대 약 24회(질문 3개 x (전 1회 + 후 1회) x 최대 4회, REPEATS=2면 두 배) + 판단(Decision API) 약 20회, REPEATS=2면 약 40회. 바꿀 조건은 정확히 하나: chunker | mode | k | reranker | decompose | corroboration
from rag_lab.corpus import fixed_chunks, section_chunks
from rag_lab.evaluation import evaluate_retrieval, recall_by_type
from rag_lab.pipeline import retrieve

MY_CHANGE = {"reranker": "off"}       # 예: {"decompose": "off"}  {"corroboration": "off"}  {"mode": "bm25"}  {"k": 1}
MY_PREDICTION = "같음"                 # 질문 3개 x REPEATS회에서 '기대 상태 + 검증 통과' 합계가 "나아짐" | "같음" | "나빠짐"
REPEATS = 1                            # 전과 후를 각각 몇 번 돌리나. 1회는 노이즈와 효과를 못 가른다. 비용을 낼 수 있으면 2로 바꿔 같은 질문의 흔들림을 보라
BASE = {"chunker": "section", "mode": "fusion", "reranker": "on", "k": 3, "decompose": "on", "corroboration": "on"}
CHUNKERS = {"section": section_chunks, "fixed": fixed_chunks}


def is_good(expected_status, rec):
    return rec["status"] == expected_status and rec["validation"] == "passed"


valid = {"chunker": set(CHUNKERS), "mode": {"bm25", "dense", "hybrid", "fusion"}, "reranker": {"on", "off"}, "k": {1, 2, 3, 4, 5},
         "decompose": {"on", "off"}, "corroboration": {"on", "off"}}
(change_key, change_value), = MY_CHANGE.items() if len(MY_CHANGE) == 1 else [(None, None)]
if change_key not in valid or change_value not in valid[change_key] or change_value == BASE[change_key]:
    print("MY_CHANGE는 항목이 정확히 하나이고 기준선과 다른 값이어야 합니다. 기준선:", BASE, "| 바꿀 수 있는 항목과 값:", {k: sorted(v) for k, v in valid.items()})
else:
    configs = {"전": BASE, "후": {**BASE, change_key: change_value}}

    def build(config):
        """설정 하나로 검색기와 재순위기를 만든다. 전과 후가 같은 함수를 지난다."""
        index_ = Retriever(search_index.documents, chunker=CHUNKERS[config["chunker"]],
                           embeddings=embeddings if config["mode"] != "bm25" else None, mode=config["mode"],
                           corroboration=config["corroboration"] == "on")
        return index_, reranker if config["reranker"] == "on" else None

    built = {name: build(config) for name, config in configs.items()}
    hard_cases = load_cases(ROOT / "data/cases_hard.jsonl") + load_cases(ROOT / "data/cases_hard2.jsonl")
    hard_gold = {**load_gold(ROOT / "data/gold_hard.jsonl"), **load_gold(ROOT / "data/gold_hard2.jsonl")}  # 측정에만 쓴다. 검토 전 초안이다.
    hard_scopes = {case["id"]: scope_of(case) for case in hard_cases}
    orders = {"전": {}, "후": {}}      # 질문마다 매긴 상위 50개 순서. 전후 비교에 쓴다.

    def measure(name, k):
        """ask가 쓰는 pipeline.retrieve와 같은 경로(pool 50)로 순위를 매겨 문서 recall을 잰다. 분해 질의는 넘기지 않는다."""
        index_, ranker = built[name]

        def rank(query, query_scope):
            key = (query, query_scope.entity, query_scope.as_of)
            if key not in orders[name]:
                orders[name][key] = retrieve(query, retriever=index_, scope=query_scope, reranker=ranker, pool=50, top_n=50).chunk_ids
            return orders[name][key]
        return evaluate_retrieval(index_, hard_cases, hard_gold, k=k, rank=rank)

    def unreachable(row):
        """정답 문서 중 호출자의 범위 밖(고객, 기준일)에 있는 것이 있으면 어떤 설정으로도 못 찾는 행이다."""
        return any(not search_index.documents[doc].applies(hard_scopes[row["case_id"]]) for doc in row["expected_docs"])

    gold_rows = {name: measure(name, config["k"]) for name, config in configs.items()}
    first_rows = {name: measure(name, 1) for name in configs}
    print("바뀐 조건:", {change_key: (BASE[change_key], change_value)})
    print(f"{'유형':12s} {'전 recall':10s} {'후 recall':10s}")
    fmt = lambda v: "-" if v is None else f"{v:.2f}"
    after_by_type = {r["type"]: r for r in recall_by_type(gold_rows["후"])}
    for r in recall_by_type(gold_rows["전"]):
        print(f"{r['type']:12s} {fmt(r['recall']):10s} {fmt(after_by_type[r['type']]['recall']):10s}")
    for name, rows in gold_rows.items():
        missed = [row for row in rows if row["missing_docs"]]
        fixed = [row for row in missed if unreachable(row)]
        print(f"[{name}] 놓친 행 {len(missed)}개 = 범위 안 {len(missed) - len(fixed)}개 + 범위 밖 정답 {len(fixed)}개 (질문 {len(hard_cases)}개, k={configs[name]['k']})")
        for row in missed:
            print(f"     {row['case_id']:30s} 놓친 문서 {row['missing_docs']}{'  <- 범위 밖 정답' if unreachable(row) else ''}")
    for name, rows in first_rows.items():
        missed = [row for row in rows if row["missing_docs"]]
        fixed = sum(unreachable(row) for row in missed)
        print(f"참고 [{name}] 1위 문서만 볼 때(k=1) 놓친 행 {len(missed)}개 = 범위 안 {len(missed) - fixed}개 + 범위 밖 정답 {fixed}개")
    changed = sum(orders["전"][key] != orders["후"][key] for key in orders["전"])
    print(f"전과 후에서 상위 50개 순서가 달라진 질문: {changed}/{len(hard_cases)}개")
    if change_key in ("decompose", "corroboration"):
        print("분해 질의는 모델이 만들어야 해서 이 검색 표에 반영되지 않습니다. 분해의 효과는 아래 ask 비교에서만 봅니다." if change_key == "decompose" else
              "재순위가 켜져 있으면 조각 32개가 모두 후보에 들어가 순서를 재순위가 정합니다. 순서가 같다면 보강도 탓이 아니라 이 구조 탓입니다.")
    if change_key == "k":
        print("k는 검색 측정에만 쓰이는 값이라 ask 결과는 달라지지 않습니다. 전후 ask 비교는 하지 않습니다.")
        report_guess(MY_PREDICTION, "비교하지 않음")
    elif not my_items:
        print("내 질문 셀에 질문이 없어 ask 비교를 하지 않습니다.")
        report_guess(MY_PREDICTION, "비교하지 않음")
    else:
        # 같은 셀에서 전과 후를 번갈아 돌린다. 앞 셀의 기록을 재사용하지 않는다.
        announce(f"전후 ask_two_phase 번갈아 {REPEATS}회씩", model_calls=4 * len(my_items) * REPEATS * 2)
        runs = {"전": [], "후": []}
        for repeat in range(1, REPEATS + 1):
            for name, config in configs.items():
                index_, ranker = built[name]
                runs[name].append([ask_two_phase(question, retriever=index_, scope=my_scope, model=model, decider=decider, reranker=ranker,
                                                 budget=Budget(), decompose=config["decompose"] == "on")
                                   for _, question, _ in my_items])
        print()
        for q, (label, question, expected_status) in enumerate(my_items):
            print(f"[{label}] 기대 {expected_status}")
            for name in ("전", "후"):
                for repeat, rows in enumerate(runs[name], 1):
                    rec = rows[q]
                    print(f"   {name}{repeat}: {rec['status']}/{rec['validation']} {'통과' if is_good(expected_status, rec) else '미통과'}"
                          f" | 인용 {cited_docs(rec)} | 질의 {len(rec['queries'])} | 검증 턴 {'돌림' if rec['verification'] == 'ran' else '건너뜀'}"
                          f" | 모델 {rec['trace']['counts'].get('model', 0)} 판단 {rec['trace']['counts'].get('decision', 0)} | {rec['trace']['elapsed_seconds']:.1f}초")
        good = {name: [sum(is_good(item[2], rec) for item, rec in zip(my_items, rows)) for rows in runs[name]] for name in runs}
        calls = {name: [sum(rec["trace"]["counts"].get("model", 0) for rec in rows) for rows in runs[name]] for name in runs}
        judged = {name: [sum(rec["trace"]["counts"].get("decision", 0) for rec in rows) for rows in runs[name]] for name in runs}
        secs = {name: [sum(rec["trace"]["elapsed_seconds"] for rec in rows) for rows in runs[name]] for name in runs}
        print()
        print(f"{'조건':4s} {'반복':4s} {'통과 행':8s} {'모델 호출':10s} {'판단 호출':10s} {'초':8s}")
        for name in runs:
            for repeat in range(REPEATS):
                print(f"{name:4s} {repeat + 1:<4d} {good[name][repeat]}/{len(my_items):<6d} {calls[name][repeat]:<10d} {judged[name][repeat]:<10d} {secs[name][repeat]:<8.1f}")
        spread = {name: max(good[name]) - min(good[name]) for name in runs}
        effect = sum(good["후"]) - sum(good["전"])
        print(f"같은 조건 안의 반복 차이(통과 행): 전 {spread['전']}, 후 {spread['후']} | 조건 차이(후 합계 - 전 합계): {effect:+d}")
        if REPEATS < 2:
            print("반복이 1회라 같은 조건 안의 차이를 알 수 없습니다. 조건 차이가 우연인지 효과인지 가를 수 없으니 REPEATS = 2로 바꿔 다시 보세요.")
        elif abs(effect) <= max(spread.values()):
            print("조건 차이가 같은 조건 안의 반복 차이보다 크지 않습니다. 이 표만으로는 조건 때문이라고 말할 수 없습니다.")
        else:
            print(f"조건 차이가 반복 차이보다 큽니다. 그래도 반복 {REPEATS}회, 질문 3개의 관찰입니다.")
        report_guess(MY_PREDICTION, "나아짐" if effect > 0 else "나빠짐" if effect < 0 else "같음")


기본값 `{"reranker": "off"}`의 한 번 출력입니다. `k=3` recall은 전후가 같고 놓친 행 4개가 모두 범위 밖 정답이지만, 상위 50개 순서는 36개 질문 모두 달라졌고 `k=1`에서 놓친 행은 7개에서 10개(늘어난 3개는 범위 안)가 됐습니다. 질문 3개는 전후 한 번씩 모두 통과해 조건 차이가 0이지만, 반복 1회라 이 0을 해석할 수 없습니다. 범위 함정 질문의 전만 질의 5개로 분해됐으나 6번 중 1번이라 재순위 탓이라 할 수 없습니다. 재순위를 켠 채 `corroboration`을 끄면 순서가 바뀐 질문이 0개였습니다(API 없이 따로 확인).

## 확인 질문

1. 2절 마지막 [바꿔보기]에서 알파와 베타 문서 목록의 차이와 공통 문서를 읽으세요. 고객이 도구 인자였다면 모델이 할 수 있었을 일을 구체적 질문 하나로 쓰세요.
2. 8절 [깨뜨리기]의 '현재 턴에 없는 인용 ID' 오류는 어느 목록과 대조해 나옵니까? 3단계에서 `RetrievalContext`를 새로 만들지 않았다면 앞 단계 ID를 인용한 답을 이 검사는 어떻게 다룹니까?
3. 6절 [깨뜨리기]의 `ValueError`는 7절의 일곱 층 중 어디입니까? `loaded_skills`에 없는 이름이 답변의 `skill`에 있는 경우와 같은 층입니까?
4. 9절 [깨뜨리기] 출력을 보고, 같은 질문 다섯 번 중 두 번이 예산 초과였다면 실패 기록을 지운 표와 남긴 표의 성공률은 각각 얼마이며 어느 쪽이 운영을 닮았습니까?
5. 11절 층 후보 표에서 `rerank-failed` 경고가 든 행의 층 후보는 어떻게 바뀝니까? 그 행의 문서 순서는 어떤 순서이고 04의 어느 표와 비교합니까?
6. 10절 출력의 게이트 세 줄에서, Decision API가 세 번 모두 실패했다면 각 줄과 `검증 턴`이 어떻게 달라질지 예측하세요. 그 흔적은 기록의 어디에 남습니까?
7. 직접 해보기에서 `reranker`는 켠 채 `corroboration`만 끄면 `전 recall`과 `후 recall`이 같을지, 04 출력의 `범위가 허용한 조각` 수와 후보 상한으로 쓰세요. 어떤 조건이면 틀립니까?
8. 직접 해보기 비교 셀의 `조건 차이`와 `반복 차이` 줄(본인 출력)에서 말할 수 있는 문장과 없는 문장을 하나씩 쓰세요. 반복을 2회로 올리면 새로 말할 수 있는 것은?

## 정리

- 다섯 단계를 직접 이었습니다: 검색 도구 → `create_deep_agent` → Skill 연결 → `validate_answer` → 기록 읽기. 여섯째로 `ask_two_phase`가 분해 게이트 → 미리 검색 → 빠른 답변 → 인용 관련성 → 추가 확인 게이트 → 검증 턴을 이었습니다.
- 실패는 검색, 범위, Skill, 답변, 검증, 예산, 판단 중 어느 층인지 가른 뒤에 고칩니다. 게이트가 기본값으로 떨어진 흔적은 `fallback`에 남습니다.
- 비교는 하나만 바꾸고, 같은 사례로, 전후를 모두 보입니다.
- 질문 3개, 반복 1회(기본)는 관찰이지 통계가 아니고 노이즈와 효과를 못 가릅니다.